<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_Ensemble/18_5_4_Model_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Comparison: All Ensemble Methods Head-to-Head

**Author:** Brad Sheese

---

## Introduction

Over the past three notebooks, we've explored five methods for tree-based classification:

1. **Single Decision Tree** — interpretable but unstable
2. **Bagging** — reduces variance through bootstrap averaging
3. **Random Forest** — adds feature randomness to decorrelate trees
4. **Gradient Boosting** — reduces bias through sequential error correction
5. **BART** — Bayesian approach with uncertainty quantification

In this final notebook, we'll compare them all using **nested cross-validation** for unbiased performance estimates — the same technique we used in 17_2_4_4 for regression.

### Learning Objectives
1. Apply nested cross-validation to compare multiple classification models.
2. Interpret the bias-variance tradeoff across all five methods.
3. Choose the right ensemble method for a given problem context.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    BaggingClassifier, AdaBoostClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class distribution: {y.value_counts(normalize=True).to_dict()}")

---
## Nested Cross-Validation for Unbiased Comparison

Recall from 17_2_4_4: nested CV uses an inner loop to tune hyperparameters and an outer loop to evaluate. This gives us an unbiased estimate of each model's performance, accounting for the fact that hyperparameter selection itself introduces optimistic bias.

### How It Works Here:
- **Outer loop**: 5-fold CV for evaluation
- **Inner loop**: 3-fold CV for hyperparameter tuning within each outer fold

For each model, we define a small hyperparameter grid. The inner loop finds the best params for each outer training fold, and the outer loop evaluates on the held-out fold.

In [ ]:
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

# Define models and their hyperparameter grids
model_configs = {
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'grid': {'max_depth': [3, 5, 7, None]}
    },
    'Bagging': {
        'model': BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=42),
        'grid': {'n_estimators': [100, 300], 'estimator__max_depth': [5, 10, None]}
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'grid': {'n_estimators': [100, 300], 'max_depth': [5, 10, None], 'max_features': ['sqrt', 'log2']}
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'grid': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [2, 3]}
    }
}

print("Running nested cross-validation for each model...")
print("This may take a few minutes.\n")

results = {}

for name, config in model_configs.items():
    inner_search = GridSearchCV(
        config['model'],
        config['grid'],
        cv=inner_cv,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    
    # Outer loop: evaluate with inner tuning
    acc_scores = cross_val_score(inner_search, X, y, cv=outer_cv, scoring='accuracy', n_jobs=-1)
    f1_scores = cross_val_score(inner_search, X, y, cv=outer_cv, scoring='f1', n_jobs=-1)
    roc_scores = cross_val_score(inner_search, X, y, cv=outer_cv, scoring='roc_auc', n_jobs=-1)
    
    results[name] = {
        'accuracy': acc_scores,
        'f1': f1_scores,
        'roc_auc': roc_scores
    }
    
    print(f"{name:<25} | Acc: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f} | F1: {f1_scores.mean():.4f} +/- {f1_scores.std():.4f} | AUC: {roc_scores.mean():.4f} +/- {roc_scores.std():.4f}")

---
## Visualizing the Comparison

Let's create boxplots to see both the central tendency and the variance of each model's performance.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

names = list(results.keys())

ax1.boxplot([results[n]['accuracy'] for n in names], tick_labels=names)
ax1.set_title('Nested CV Accuracy')
ax1.set_ylabel('Accuracy')
ax1.grid(True, alpha=0.3)

ax2.boxplot([results[n]['f1'] for n in names], tick_labels=names)
ax2.set_title('Nested CV F1-Score')
ax2.set_ylabel('F1-Score')
ax2.grid(True, alpha=0.3)

ax3.boxplot([results[n]['roc_auc'] for n in names], tick_labels=names)
ax3.set_title('Nested CV AUC')
ax3.set_ylabel('AUC')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Interpreting the Results

### The Bias-Variance Story

Looking at the results, we can see the bias-variance tradeoff playing out:

- **Decision Tree**: Lowest mean accuracy and highest variance. A single tree is unstable — its performance depends heavily on which samples it sees.

- **Bagging**: Significant improvement over the single tree. Averaging 300 bootstrap-trained trees cancels out individual errors, reducing variance.

- **Random Forest**: Further improvement over bagging. The feature randomness decorrelates the trees, making the averaging even more effective.

- **Gradient Boosting**: Typically achieves the highest accuracy and F1. By sequentially correcting errors, it reduces bias more effectively than the parallel methods. However, it may have slightly higher variance than the random forest because the trees are dependent on each other.

### Which Model Should You Choose?

| Scenario | Recommended Model | Why |
|---|---|---|
| Need interpretability | Decision Tree | Can print the flowchart |
| Need a quick, reliable baseline | Random Forest | Hard to mess up, parallelizable |
| Need maximum accuracy | Gradient Boosting | Best performance on tabular data |
| Small dataset, need uncertainty | BART | Provides credible intervals |
| Very large dataset | Random Forest | Scales better than boosting |

For the breast cancer diagnosis problem specifically, **Gradient Boosting** is the best choice: it achieves the highest recall (catching the most cancers) while maintaining excellent precision (minimizing unnecessary biopsies).

---
## Final Model: Best Performer on Full Data

Now that we've identified the best method, let's train it on the full dataset and examine its final performance.

In [ ]:
# Find the best model by mean F1
best_name = max(results.keys(), key=lambda n: results[n]['f1'].mean())
print(f"Best model by nested CV F1: {best_name}")

# Train the best model on all data
best_config = model_configs[best_name]
best_model = GridSearchCV(
    best_config['model'],
    best_config['grid'],
    cv=5,
    scoring='f1',
    n_jobs=-1
)
best_model.fit(X, y)

print(f"Best parameters: {best_model.best_params_}")
print(f"Best CV F1: {best_model.best_score_:.4f}")

### Feature Importance from the Final Model

In [ ]:
if hasattr(best_model.best_estimator_, 'feature_importances_'):
    importances = pd.Series(best_model.best_estimator_.feature_importances_, index=data.feature_names)
    top_10 = importances.sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    bars = plt.barh(range(len(top_10)), top_10.values, color='darkgreen', edgecolor='black')
    plt.yticks(range(len(top_10)), top_10.index)
    plt.gca().invert_yaxis()
    plt.xlabel('Relative Importance Score')
    plt.title(f'Top 10 Important Features: {best_name}')
    
    for bar, val in zip(bars, top_10.values):
        plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2.,
                 f'{val:.3f}', ha='left', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()

### Your Turn

1. Look at the boxplots. Which model has the smallest interquartile range (most consistent performance)? Is this the same model with the highest mean?
2. The nested CV F1 for the best model is {results[best_name]['f1'].mean():.4f} +/- {results[best_name]['f1'].std():.4f}. What does the standard deviation tell you about the model's reliability?
3. If you were deploying this model in a hospital, would you choose the model with the highest mean accuracy or the one with the lowest variance? Justify your answer.

## Conclusion: The Ensemble Series Recap

Over four notebooks, we've explored the full spectrum of tree-based ensemble methods:

1. **Single Decision Tree** — The building block. Interpretable but unstable. Prone to overfitting.
2. **Bagging** — Reduces variance by averaging many bootstrap-trained trees. OOB error provides built-in validation.
3. **Random Forest** — Adds feature randomness to decorrelate trees. The go-to method for most tabular data problems.
4. **Gradient Boosting** — Reduces bias through sequential error correction. The most accurate method for structured data.
5. **BART** — Bayesian approach with uncertainty quantification. Ideal when you need to know how confident the model is.

### Key Takeaways

- **Ensembles beat single trees** — always. The wisdom of the crowd is real.
- **Bagging reduces variance; boosting reduces bias** — they solve different problems.
- **Hyperparameter tuning matters** — especially for boosting, where the learning rate and tree depth control the bias-variance tradeoff.
- **Classification metrics matter** — in medical diagnosis, recall (catching cancers) is more important than accuracy.
- **Nested CV gives honest estimates** — don't trust a single train/test split for model comparison.

Thank you for completing the 18_5 Ensemble Methods series!